In [1]:
!pip install tensorflow opencv-python matplotlib

In [2]:
#importing dependencies
import cv2
import os
import random
import numpy as np
import matplotlib.pyplot as plt


In [3]:
#import tensorflow - functional API

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, Conv2D, Dense, MaxPooling2D, Input, Flatten
import tensorflow as tf


In [4]:
gpu = tf.config.experimental.list_logical_devices("GPU")
gpu
#since running colab on cloudserver without gpu

[]

In [5]:
#setp paths

POS_PATH = os.path.join('data', 'positive')
NEG_PATH = os.path.join('data', 'negative')
ANC_PATH = os.path.join('data', 'anchor')


os.makedirs(POS_PATH)
os.makedirs(NEG_PATH)
os.makedirs(ANC_PATH)

In [6]:
#getting anchors and positives
from google.colab import files

# This will open a "Choose Files" button
uploaded = files.upload()

# Optional: Print the name of the uploaded file(s)
for filename in uploaded.keys():
    print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')


Saving lfw.zip to lfw.zip
User uploaded file "lfw.zip" with length 296875 bytes


In [7]:
!unzip /content/lfw.zip -d /content/lfw


Archive:  /content/lfw.zip
   creating: /content/lfw/lfw/Ataollah_Mohajerani/
  inflating: /content/lfw/lfw/Ataollah_Mohajerani/Ataollah_Mohajerani_0001.jpg  
   creating: /content/lfw/lfw/Atiabet_Ijan_Amabel/
  inflating: /content/lfw/lfw/Atiabet_Ijan_Amabel/Atiabet_Ijan_Amabel_0001.jpg  
   creating: /content/lfw/lfw/Atsushi_Sato/
  inflating: /content/lfw/lfw/Atsushi_Sato/Atsushi_Sato_0001.jpg  
   creating: /content/lfw/lfw/Audrey_Lacroix/
  inflating: /content/lfw/lfw/Audrey_Lacroix/Audrey_Lacroix_0001.jpg  
   creating: /content/lfw/lfw/Audrey_Sauret/
  inflating: /content/lfw/lfw/Audrey_Sauret/Audrey_Sauret_0001.jpg  
   creating: /content/lfw/lfw/Augustin_Calleri/
  inflating: /content/lfw/lfw/Augustin_Calleri/Augustin_Calleri_0001.jpg  
  inflating: /content/lfw/lfw/Augustin_Calleri/Augustin_Calleri_0002.jpg  
  inflating: /content/lfw/lfw/Augustin_Calleri/Augustin_Calleri_0003.jpg  
  inflating: /content/lfw/lfw/Augustin_Calleri/Augustin_Calleri_0004.jpg  
   creating: /conte

FETCHING ANCHORS AND POSITIVES FROM WEBCAM USING JS SCRIPT

In [9]:
for directory in os.listdir("/content/lfw/lfw"):
  print(directory)
  for file in os.listdir(os.path.join('lfw','lfw',directory)):
    ex_path = os.path.join('lfw','lfw',directory,file)
    new_path = os.path.join(NEG_PATH,file)
    os.replace(ex_path,new_path)

Augusto_Pinochet
Barbara_Boxer
Azra_Akin
Augustin_Calleri
Audrey_Sauret
Azmi_Bishara
Augusto_Roa_Bastos
Barbara_Bodine
Avril_Lavigne
Barbara_De_Brun
Barbara_Brezigar
Audrey_Lacroix
Barbara_Bach
Atsushi_Sato
Bak_Chang-Ryun
Aung_San_Suu_Kyi
Barbara_Becker
Austin_Kearns
Atiabet_Ijan_Amabel
Ataollah_Mohajerani
Baburam_Bhattari
Babe_Ruth


In [18]:
#COLLECTING ANCOHOR AND POSITIVE USING JS SCRIPT

In [11]:
import uuid

In [22]:
import os
import time
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def continuous_capture(quality=0.8):
  # Define your two target folders

  # 1. Initialize the camera stream and UI elements once
  js_init = Javascript('''
  const constraints = { video: { width: { ideal: 250 }, height: { ideal: 250 } } };

  async function initCamera() {
    const div = document.createElement('div');
    div.id = 'camera-container';
    const instruction = document.createElement('p');
    instruction.innerHTML = 'Press <b>"a"</b> for Folder A | Press <b>"b"</b> for Folder B<br>Press <b>"Enter"</b> to Exit';
    div.appendChild(instruction);

    const video = document.createElement('video');
    video.id = 'camera-video';
    video.style.display = 'block';

    // Store stream globally so we can access it across function calls
    window.cameraStream = await navigator.mediaDevices.getUserMedia(constraints);

    document.body.appendChild(div);
    div.appendChild(video);
    video.srcObject = window.cameraStream;
    await video.play();

    google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  }
  ''')
  display(js_init)
  eval_js('initCamera()')

  # 2. JavaScript helper function to wait for a single keypress event
  js_capture = Javascript('''
  async function waitForKey() {
    let pressedKey = '';

    await new Promise((resolve) => {
      const handleKeyDown = (event) => {
        const key = event.key.toLowerCase();

        // Listen for choices OR the exit key (Enter)
        if (key === 'a' || key === 'p' || event.key === 'Enter') {
          pressedKey = event.key; // Keep original case to distinguish 'Enter'
          document.removeEventListener('keydown', handleKeyDown);
          resolve();
        }
      };
      document.addEventListener('keydown', handleKeyDown);
    });

    // If Enter was pressed, shut down the stream and clean up UI
    if (pressedKey === 'Enter') {
      window.cameraStream.getVideoTracks()[0].stop();
      document.getElementById('camera-container').remove();
      return { action: 'exit' };
    }

    // Otherwise, snap a frame
    const video = document.getElementById('camera-video');
    const canvas = document.createElement('canvas');
    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;
    canvas.getContext('2d').drawImage(video, 0, 0);

    return {
      action: 'capture',
      key: pressedKey.toLowerCase(),
      image: canvas.toDataURL('image/jpeg', 0.8)
    };
  }
  ''')
  display(js_capture)

  print("Camera active. Start pressing keys...")

  # 3. Main Python Loop to handle infinite capturing
  while True:
    result = eval_js('waitForKey()')

    if result['action'] == 'exit':
      print("Exited successfully. Camera turned off.")
      break

    chosen_key = result['key']
    data = result['image']

    # Generate unique filenames using timestamps so files don't overwrite each other
    timestamp = int(time.time() * 1000)

    if chosen_key == 'a':
      filename = os.path.join(ANC_PATH, '{}.jpg'.format(uuid.uuid1()))
      #print(f"Captured: Saved to Folder A ({filename})")
    elif chosen_key == 'p':
      filename = os.path.join(POS_PATH, '{}.jpg'.format(uuid.uuid1()))
      #print(f"Captured: Saved to Folder B ({filename})")

    # Save image binary
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
      f.write(binary)

# Run the live stream loop
continuous_capture()


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Camera active. Start pressing keys...
Captured: Saved to Folder A (data/anchor/c3d2a00a-a5da-11f1-b8d7-0242ac1c000c.jpg)
Captured: Saved to Folder A (data/anchor/c4183bec-a5da-11f1-b8d7-0242ac1c000c.jpg)
Captured: Saved to Folder A (data/anchor/c4662960-a5da-11f1-b8d7-0242ac1c000c.jpg)
Captured: Saved to Folder A (data/anchor/c4bf2bdc-a5da-11f1-b8d7-0242ac1c000c.jpg)
Captured: Saved to Folder A (data/anchor/c510794c-a5da-11f1-b8d7-0242ac1c000c.jpg)
Captured: Saved to Folder A (data/anchor/c542dae0-a5da-11f1-b8d7-0242ac1c000c.jpg)
Captured: Saved to Folder A (data/anchor/c594299a-a5da-11f1-b8d7-0242ac1c000c.jpg)
Captured: Saved to Folder A (data/anchor/c61b8b56-a5da-11f1-b8d7-0242ac1c000c.jpg)
Captured: Saved to Folder A (data/anchor/c66ce6cc-a5da-11f1-b8d7-0242ac1c000c.jpg)
Captured: Saved to Folder A (data/anchor/c6d18ee2-a5da-11f1-b8d7-0242ac1c000c.jpg)
Captured: Saved to Folder A (data/anchor/c71b3754-a5da-11f1-b8d7-0242ac1c000c.jpg)
Captured: Saved to Folder A (data/anchor/c7706918

In [23]:
len(os.listdir(ANC_PATH))


88

In [24]:
anchor = tf.data.Dataset.list_files(ANC_PATH+'/*.jpg').take(50)
positive = tf.data.Dataset.list_files(POS_PATH+'/*.jpg').take(50)
negative = tf.data.Dataset.list_files(NEG_PATH+'/*.jpg').take(50)


In [29]:
anchor.as_numpy_iterator().next()

b'data/anchor/cc824b4c-a5da-11f1-b8d7-0242ac1c000c.jpg'

In [26]:
def preprocess(filename):
  byte_img = tf.io.read_file(filename)
  #loading img in jpeg
  img = tf.io.decode_jpeg(byte_img)
  #resizing to 100x100 as original model uses 105x05
  img = tf.image.resize(img(100,100))
  img= img/255.0
  return img


In [1]:
tf.ones(len(anchor))

NameError: name 'tf' is not defined

In [ ]:
#create labelled data of (anc,pos) & (anc,neg)

positives = tf.data.Dataset.zip((anchor,positive,tf.data.Dataset.from_tensor_slices(tf.ones(len(anchor)))))
negatives = tf.data.Dataset.zip((anchor,negative,tf.data.Dataset.from_tensor_slices(tf.zeros(len(anchor)))))
